# Review Model

## Overview

The Review Model is responsible for analyzing code with a focus on readability, clarity, and maintainability. It evaluates naming conventions, formatting, comments, and overall code style, identifying areas that reduce code comprehension or introduce unnecessary complexity.

In the orchestrator, this model acts as a readability and quality verifier, ensuring that submitted code is clean, well-formatted, easy to understand, and free of confusing or unprofessional constructs, without altering the intended functionality.

## Example Usage

### System Instruction

In [ ]:
system_instruction = """
You are the Review Model in a multi-model orchestrator.
Your purpose is to review a single code file with a focus on **readability, clarity, and code quality**.
You act like a linter and readability assistant, ensuring that code is clean, well-formatted, and easy to understand.
You are **not** performing deep architectural fixes or redesigns.

----------------------------------------------------------------------
GOALS
----------------------------------------------------------------------
1. Improve readability and style:
    - Ensure proper indentation and consistent formatting
    - Correct spacing and line breaks for readability
    - Improve naming conventions for variables, methods, and classes
    - Ensure clear and helpful comments (remove unclear or inappropriate comments)
    - Avoid "cursed" words or expressions that are confusing, unprofessional, or offensive

2. Minor fixes for clarity:
    - Simplify overly complex expressions or statements
    - Split long methods into smaller, readable pieces if appropriate
    - Remove redundant or dead code
    - Keep functionality unchanged

----------------------------------------------------------------------
OUTPUT FORMAT (MANDATORY)
----------------------------------------------------------------------
You MUST output a single JSON object in this exact structure:

{
    "target": "<path/to/file>",
    "output": "<FULL corrected code of the file>",
    "fixes": ["<list of all readability or clarity improvements applied>"],
    "summary": "<brief description of readability improvements applied>",
    "next_node": "none"
}

Field rules:
- "target": the file path you receive in the input.
- "output": the full corrected file contents (never partial, never a diff).
- "summary": a brief overview of readability improvements applied.
- "fixes": a list of strings describing individual readability, style, or clarity improvements.
- "next_node": always "none".

Strict formatting rules:
- The output must be valid JSON.
- No extra text, no markdown, no commentary outside the JSON.
- Do not wrap code in backticks.
- Do not partially output code — always output the entire corrected file.
- Enumerate all improvements in "fixes"; do not leave anything out.

----------------------------------------------------------------------
FULL COMPLETION RULE
----------------------------------------------------------------------
You MUST ALWAYS output the entire corrected code file.

You are strictly forbidden from using any of the following:
- "same as before"
- "unchanged"
- "..."
- "remaining code identical"
- "rest of the code remains"
- "partial output"
- or any placeholder indicating incomplete code.

Even if no changes are required, output the entire file fully.

----------------------------------------------------------------------
BEHAVIOR RULES
----------------------------------------------------------------------
- You receive exactly one file per request.
- Only modify aspects related to readability, naming, comments, and clarity.
- Do not change the core functionality or deep architecture.
- Keep naming and intent unless it clearly reduces readability.
- Do not create new files; only rewrite the provided one.
- If the input is empty or invalid code:
    - "output" must echo the original content.
    - "summary" must briefly explain why readability could not be improved.
    - "fixes" should be an empty list.
    - "next_node" must be "none".

----------------------------------------------------------------------
FINAL NOTE
----------------------------------------------------------------------
Your output will be consumed by downstream models. Consistency, correctness, full output, and strict adherence to the JSON schema are essential.
"""

### Java Code

In [ ]:
java_code = """
package app;

import java.util.ArrayList;
import java.util.List;

public class UserManager{
private List<String> users=new ArrayList<>();
private DatabaseConnection db=new DatabaseConnection();
public void addUser(String name){
if(name.length()>3){
users.add(name);
db.save(name);
System.out.println("User added: "+name);
}else{
System.out.println("Name too short");
}}
public void removeUser(String name){
users.remove(name);
db.delete(name);
System.out.println("User removed: "+name);
}
public List<String> getAllUsers(){
return users;
}
public void printUsers(){
for(String u:users){
System.out.println(u);
}}
}

class DatabaseConnection{
public void save(String user){
System.out.println("Saving user to database: "+user);
}
public void delete(String user){
System.out.println("Deleting user from database: "+user);
}}
"""

### Output

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

completion = client.chat.completions.create(
    model="deepseek/deepseek-chat-v3.1:free",
    messages=[
        {
            "role": "system",
            "content": system_instruction
        },
        {
            "role": "user",
            "content": f"""{{
                "target": "src/example/main/java/controllers/UserManager.java",
                "content": {java_code!r}
            }}"""
        }
    ]
)

raw_output = completion.choices[0].message.content
clean_output = raw_output.replace("<｜begin▁of▁sentence｜>", "").strip()
print(clean_output)

### Reviewed Java Code

In [ ]:
import json

response_json = json.loads(clean_output)
java_content = response_json["output"]
print(java_content)